In [1]:
from pathlib import Path 

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

ROOT = Path.cwd() 
while not (ROOT / "src" / "data" / "synthetic" / "run.py" ).exists() and ROOT != ROOT.parent: 
    ROOT = ROOT.parent 

PARQUET = ROOT / "src" / "data" / "synthetic" / "parquet"

pre_v1 = pd.read_parquet(PARQUET / "claims_pre_v1.parquet")
log = pd.read_parquet(PARQUET / "claims_v1_log.parquet")

# 2016-2021
print(f"claims_pre_v1: {len(pre_v1):}") 

#2022-2024
print(f"claims_v1_log: {len(log):}") 

claims_pre_v1: 46775
claims_v1_log: 23225


# p27 - propensity score

e(X) = P(T=1 | X)
- decision T 
- X: features 
- e(X): with features X, what is the probability of decision making "scrap"

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

feat = ["repair_to_value_ratio","damage_severity","vehicle_age_years","mileage","vehicle_type","damage_location","damage_type"]

Xd = pd.get_dummies(log[feat], drop_first=True).astype(float)
Xd = Xd.fillna(Xd.median(numeric_only=True))

e = LogisticRegression(max_iter=1000).fit(Xd, log.model_v1_decision).predict_proba(Xd)[:, 1]
print(f"propensity-to-scrap AUC = {roc_auc_score(log.model_v1_decision, e):.3f}")